# Physical AI — A*経路計画と模倣学習によるロボット経路追従

**提出用整理版（STEP 5-1）**。元ノートブックの検証済みコードから、STEP 4-1〜4-5 ver.2を実行順に抽出しました。古い試行版（STEP 3-1〜3-5）は元ノートブックに保存されています。

## 実行方法

1. Google Colabでこのノートブックを開き、ランタイムを接続します（CPUで実行可能）。
2. **上から順番に**コードセルを実行します。各セルが次のセルに必要なファイル・関数を作成します。
3. STEP 4-1と4-4、4-5ではMP4を生成します。動画出力にはColabの `ffmpeg` が必要です。
4. Colabの一時領域はセッション終了時に消去されるため、必要なCSV、PNG、MP4、`.pth`をダウンロードしてください。

**実装範囲：** A*が障害物を考慮して経路を計画し、MLPが教師の経路追従制御を模倣します。AIは距離センサを直接使わず、入力は `goal_distance` と `heading_error` の2変数です。強化学習は実装していません。

**備考：** この整理版は、コードの静的構文チェック、およびColabでの全セル再実行による再現確認を行いました。


## STEP 4-1：滑らかな教師コントローラと基準環境の走行

このセルから順に実行してください。2つ目以降のセルは、前のSTEPのセルが完了してから実行してください。


In [ ]:

# ============================================================
# STEP 4-1 : 滑らかな教師コントローラの実装
# 現在の5つの障害物配置を維持し、A*で計画した経路を滑らかに走行させる。
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import heapq
import math
import csv
import os
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import Video, display

# ------------------------------------------------------------
# 1. Environment
# ------------------------------------------------------------
world_size = 10.0

obstacles = [
    (1.4, 2.6, 3.0, 0.7),
    (7.2, 4.1, 6.0, 0.7),
    (3.6, 5.6, 2.0, 0.7),
    (7.2, 7.1, 3.0, 0.7),
    (2.8, 8.6, 3.0, 0.7)
]

start = np.array([1.0, 1.0])
goal = np.array([9.0, 9.0])

robot_radius = 0.25
safety_margin = 0.30

# Physical safety clearance
clearance = robot_radius + safety_margin  # 0.55 m

# Slightly larger margin for A* planning
planning_clearance = clearance + 0.02

grid_resolution = 0.20

dt = 0.05
fps = 20
max_steps = 2000

max_speed = 0.40
max_turn_speed = 1.20
goal_tolerance = 0.35

sensor_angles = [
    np.pi / 4,
    0.0,
    -np.pi / 4
]
sensor_max_range = 3.0

# ------------------------------------------------------------
# 2. Collision checking
# ------------------------------------------------------------
def is_free(point, margin=clearance):
    x, y = point

    if (
        x < margin or y < margin
        or x > world_size - margin
        or y > world_size - margin
    ):
        return False

    for cx, cy, w, h in obstacles:
        if (
            cx - w/2 - margin <= x <= cx + w/2 + margin
            and
            cy - h/2 - margin <= y <= cy + h/2 + margin
        ):
            return False

    return True


# ------------------------------------------------------------
# 3. A* path planning
# ------------------------------------------------------------
def astar(start, goal):

    def to_grid(point):
        return tuple(
            np.round(point / grid_resolution).astype(int)
        )

    def to_world(node):
        return np.array(node, dtype=float) * grid_resolution

    start_node = to_grid(start)
    goal_node = to_grid(goal)

    directions = [
        (dx, dy)
        for dx in [-1, 0, 1]
        for dy in [-1, 0, 1]
        if dx != 0 or dy != 0
    ]

    queue = [(0.0, start_node)]
    costs = {start_node: 0.0}
    parents = {}

    while queue:

        _, current = heapq.heappop(queue)

        if current == goal_node:
            break

        for dx, dy in directions:

            neighbor = (
                current[0] + dx,
                current[1] + dy
            )

            if not is_free(
                to_world(neighbor),
                planning_clearance
            ):
                continue

            # Prevent diagonal corner cutting
            if dx != 0 and dy != 0:
                side1 = (current[0] + dx, current[1])
                side2 = (current[0], current[1] + dy)

                if (
                    not is_free(to_world(side1), planning_clearance)
                    or
                    not is_free(to_world(side2), planning_clearance)
                ):
                    continue

            new_cost = (
                costs[current] + math.hypot(dx, dy)
            )

            if new_cost < costs.get(neighbor, float("inf")):

                costs[neighbor] = new_cost
                parents[neighbor] = current

                heuristic = math.hypot(
                    neighbor[0] - goal_node[0],
                    neighbor[1] - goal_node[1]
                )

                heapq.heappush(
                    queue,
                    (new_cost + heuristic, neighbor)
                )

    if goal_node != start_node and goal_node not in parents:
        raise RuntimeError("A* could not find a safe path.")

    path = [goal_node]

    while path[-1] != start_node:
        path.append(parents[path[-1]])

    path.reverse()

    return np.array(
        [to_world(node) for node in path]
    )


path = astar(start, goal)

print("A* path points:", len(path))


# ------------------------------------------------------------
# 4. Continuous path representation
# ------------------------------------------------------------
segment_lengths = np.linalg.norm(
    np.diff(path, axis=0),
    axis=1
)

cumulative_lengths = np.concatenate(
    [[0.0], np.cumsum(segment_lengths)]
)

total_path_length = cumulative_lengths[-1]


def point_on_path(s):

    s = np.clip(s, 0.0, total_path_length)

    index = np.searchsorted(
        cumulative_lengths,
        s,
        side="right"
    ) - 1

    index = min(index, len(segment_lengths) - 1)

    segment_length = segment_lengths[index]

    if segment_length < 1e-10:
        return path[index].copy()

    t = (
        s - cumulative_lengths[index]
    ) / segment_length

    return (
        path[index]
        + t * (path[index + 1] - path[index])
    )


# ------------------------------------------------------------
# 5. Virtual distance sensors
# ------------------------------------------------------------
def sensor_distance(position, angle):

    step = 0.04

    for distance in np.arange(
        0.0,
        sensor_max_range + step,
        step
    ):

        test_point = position + distance * np.array([
            np.cos(angle),
            np.sin(angle)
        ])

        # Sensor measures distance to actual obstacle surfaces
        if (
            test_point[0] < 0
            or test_point[1] < 0
            or test_point[0] > world_size
            or test_point[1] > world_size
        ):
            return float(distance)

        for cx, cy, w, h in obstacles:
            if (
                cx - w/2 <= test_point[0] <= cx + w/2
                and
                cy - h/2 <= test_point[1] <= cy + h/2
            ):
                return float(distance)

    return sensor_max_range


def read_sensors(position, theta):

    return [
        sensor_distance(position, theta + angle)
        for angle in sensor_angles
    ]


# ------------------------------------------------------------
# 6. Smooth teacher controller
# ------------------------------------------------------------
def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


position = start.copy()
theta = 0.0

path_progress = 0.0

trajectory = [position.copy()]
heading_history = [theta]

teacher_records = []

status = "TIMEOUT"

for step_index in range(max_steps):

    # Find nearest progress point on the planned path
    candidate_s = np.linspace(
        max(0.0, path_progress - 0.25),
        min(total_path_length, path_progress + 1.50),
        75
    )

    candidate_points = np.array([
        point_on_path(s) for s in candidate_s
    ])

    distances = np.linalg.norm(
        candidate_points - position,
        axis=1
    )

    nearest_s = candidate_s[np.argmin(distances)]

    path_progress = max(
        path_progress,
        float(nearest_s)
    )

    # Pure-pursuit-style lookahead target
    lookahead = 0.16

    target_s = min(
        total_path_length,
        path_progress + lookahead
    )

    target_point = point_on_path(target_s)

    delta = target_point - position

    target_angle = math.atan2(
        delta[1],
        delta[0]
    )

    heading_error = wrap_angle(
        target_angle - theta
    )

    # Continuous angular velocity
    omega = np.clip(
        2.7 * heading_error,
        -max_turn_speed,
        max_turn_speed
    )

    # Reduce forward speed during sharp turns
    v = max_speed * max(
        0.08,
        1.0 - 0.9 * min(abs(heading_error), 1.3)
    )

    # Reduce speed near goal
    goal_distance = np.linalg.norm(
        goal - position
    )

    if goal_distance < 0.7:
        v = min(
            v,
            0.22 + 0.5 * goal_distance
        )

    # Read sensors for future imitation learning
    sensor_values = read_sensors(
        position,
        theta
    )

    # Save teacher state and action
    teacher_records.append([
        step_index * dt,
        position[0],
        position[1],
        theta,
        goal_distance,
        np.linalg.norm(delta),
        heading_error,
        sensor_values[0],
        sensor_values[1],
        sensor_values[2],
        v,
        omega
    ])

    # Differential-drive kinematic update
    next_position = position + dt * v * np.array([
        np.cos(theta),
        np.sin(theta)
    ])

    next_theta = wrap_angle(
        theta + omega * dt
    )

    # Physical collision safety check
    if not is_free(next_position, clearance):
        status = "BLOCKED"
        print(
            "BLOCKED at step",
            step_index,
            "position:",
            position
        )
        break

    position = next_position
    theta = next_theta

    trajectory.append(position.copy())
    heading_history.append(theta)

    if np.linalg.norm(goal - position) < goal_tolerance:
        status = "GOAL REACHED"
        break


trajectory = np.array(trajectory)
heading_history = np.array(heading_history)

print("Status:", status)
print("Steps:", len(trajectory) - 1)
print("Time:", round((len(trajectory) - 1) * dt, 2), "s")
print("Final position:", position)

# ------------------------------------------------------------
# 7. Save teacher dataset
# ------------------------------------------------------------
csv_filename = "step4_1_teacher_data.csv"

with open(csv_filename, "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "time",
        "x",
        "y",
        "theta",
        "goal_distance",
        "target_distance",
        "heading_error",
        "sensor_left",
        "sensor_front",
        "sensor_right",
        "v",
        "omega"
    ])

    writer.writerows(teacher_records)

print("Teacher data saved:", csv_filename)


# ------------------------------------------------------------
# 8. Generate MP4 animation
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(0, world_size)
ax.set_ylim(0, world_size)
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)

ax.set_title("STEP 4-1: Smooth Teacher Controller")

for cx, cy, w, h in obstacles:

    rect = patches.Rectangle(
        (cx - w/2, cy - h/2),
        w,
        h,
        facecolor="salmon",
        edgecolor="darkred",
        alpha=0.75
    )

    ax.add_patch(rect)

ax.plot(
    path[:, 0],
    path[:, 1],
    "--",
    color="gray",
    linewidth=1,
    alpha=0.6,
    label="A* path"
)

ax.plot(
    start[0], start[1],
    "go",
    markersize=9,
    label="Start"
)

ax.plot(
    goal[0], goal[1],
    "r*",
    markersize=14,
    label="Goal"
)

trail_line, = ax.plot(
    [], [],
    color="royalblue",
    linewidth=2.2,
    label="Teacher trajectory"
)

robot_circle = patches.Circle(
    start,
    robot_radius,
    facecolor="royalblue",
    edgecolor="navy",
    alpha=0.85
)

ax.add_patch(robot_circle)

heading_line, = ax.plot(
    [], [],
    color="navy",
    linewidth=2
)

sensor_lines = [
    ax.plot([], [], color="orange", linewidth=1, alpha=0.7)[0]
    for _ in sensor_angles
]

time_text = ax.text(
    0.02,
    0.97,
    "",
    transform=ax.transAxes,
    va="top"
)

ax.legend(loc="lower right")


def update(frame):

    p = trajectory[frame]
    angle = heading_history[frame]

    robot_circle.center = p

    trail_line.set_data(
        trajectory[:frame+1, 0],
        trajectory[:frame+1, 1]
    )

    heading_end = p + robot_radius * 1.5 * np.array([
        np.cos(angle),
        np.sin(angle)
    ])

    heading_line.set_data(
        [p[0], heading_end[0]],
        [p[1], heading_end[1]]
    )

    sensor_values = read_sensors(p, angle)

    for line, relative_angle, distance in zip(
        sensor_lines,
        sensor_angles,
        sensor_values
    ):

        sensor_angle = angle + relative_angle

        endpoint = p + distance * np.array([
            np.cos(sensor_angle),
            np.sin(sensor_angle)
        ])

        line.set_data(
            [p[0], endpoint[0]],
            [p[1], endpoint[1]]
        )

    time_text.set_text(
        f"Time: {frame * dt:.1f} s"
    )

    return (
        trail_line,
        robot_circle,
        heading_line,
        time_text,
        *sensor_lines
    )


animation = FuncAnimation(
    fig,
    update,
    frames=len(trajectory),
    interval=1000 / fps,
    blit=True
)

video_filename = "step4_1_smooth_teacher.mp4"

writer = FFMpegWriter(
    fps=fps,
    bitrate=1600
)

animation.save(
    video_filename,
    writer=writer
)

plt.close(fig)

print("Video saved:", video_filename)

display(
    Video(
        video_filename,
        embed=True,
        width=640
    )
)

# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------
print("\n===== STEP 4-1 RESULT =====")
print("Controller: Smooth Rule-Based Teacher")
print("Status:", status)
print("Trajectory points:", len(trajectory))
print("Teacher samples:", len(teacher_records))
print("Video:", video_filename)
print("Dataset:", csv_filename)


## STEP 4-2：教師データ収集・エピソード単位での分割

前のSTEPのセルが完了してから実行してください。


In [ ]:

# ============================================================
# STEP 4-2 : 教師データの自動収集
# Multiple starts, goals, and initial headings
#
# Requires STEP 4-1 to have been executed first.
# Does not overwrite the STEP 4-1 trajectory or video.
#
# 今回の収集設定
# 学習データ作成
#
# 試行回数          60回
# 最大走行時間／回  90秒
# 障害物配置        STEP 4-1と同一
# 保存対象          ゴール到達に成功した走行
# ============================================================

import numpy as np
import pandas as pd
import math
from IPython.display import display

# ------------------------------------------------------------
# 1. Data collection settings
# ------------------------------------------------------------
rng = np.random.default_rng(2026)

NUM_EPISODES = 60
MAX_STEPS_PER_EPISODE = 1800

# Same environment and controller parameters as STEP 4-1
# Uses existing:
# obstacles, astar, is_free, read_sensors, wrap_angle,
# dt, max_speed, max_turn_speed, goal_tolerance,
# clearance, grid_resolution

feature_columns = [
    "goal_distance",
    "target_distance",
    "heading_error",
    "sensor_left",
    "sensor_front",
    "sensor_right"
]

action_columns = ["v", "omega"]

all_rows = []
episode_results = []


# ------------------------------------------------------------
# 2. Local path interpolation
#    (Independent of STEP 4-1 global path variable)
# ------------------------------------------------------------
def make_path_geometry(planned_path):

    lengths = np.linalg.norm(
        np.diff(planned_path, axis=0),
        axis=1
    )

    cumulative = np.concatenate(
        [[0.0], np.cumsum(lengths)]
    )

    return lengths, cumulative


def interpolate_path(
    planned_path,
    lengths,
    cumulative,
    s
):

    total = cumulative[-1]
    s = float(np.clip(s, 0.0, total))

    index = np.searchsorted(
        cumulative,
        s,
        side="right"
    ) - 1

    index = min(index, len(lengths) - 1)

    length = lengths[index]

    if length < 1e-10:
        return planned_path[index].copy()

    t = (s - cumulative[index]) / length

    return (
        planned_path[index]
        + t * (
            planned_path[index + 1]
            - planned_path[index]
        )
    )


# ------------------------------------------------------------
# 3. Random safe start / goal positions
# ------------------------------------------------------------
def sample_safe_position():

    for _ in range(1000):

        candidate = rng.uniform(
            0.8,
            9.2,
            size=2
        )

        # A* rounds coordinates to grid points.
        # Use exact grid-aligned positions.
        candidate = (
            np.round(candidate / grid_resolution)
            * grid_resolution
        )

        if is_free(candidate, clearance + 0.10):
            return candidate

    raise RuntimeError(
        "Could not sample a safe position."
    )


def sample_episode():

    # Mix familiar and diverse start positions.
    if rng.random() < 0.35:

        s = np.array([1.0, 1.0]) + rng.uniform(
            -0.30,
            0.30,
            size=2
        )

        s = (
            np.round(s / grid_resolution)
            * grid_resolution
        )

        if not is_free(s, clearance + 0.10):
            s = sample_safe_position()

    else:
        s = sample_safe_position()

    # Most episodes aim at the original goal.
    # Others use different safe goals.
    if rng.random() < 0.75:
        g = np.array([9.0, 9.0])
    else:
        g = sample_safe_position()

    return s, g


# ------------------------------------------------------------
# 4. Run one smooth teacher episode
# ------------------------------------------------------------
def run_teacher_episode(
    episode_id,
    episode_start,
    episode_goal
):

    # Plan route with existing A* implementation
    try:
        planned_path = astar(
            episode_start,
            episode_goal
        )
    except RuntimeError:
        return "NO_PATH", [], 0

    if len(planned_path) < 2:
        return "TOO_SHORT", [], 0

    lengths, cumulative = make_path_geometry(
        planned_path
    )

    total_length = cumulative[-1]

    if total_length < 3.0:
        return "TOO_SHORT", [], 0

    position = episode_start.copy()

    # Random initial heading around the first path direction
    first_delta = (
        planned_path[1] - planned_path[0]
    )

    first_angle = math.atan2(
        first_delta[1],
        first_delta[0]
    )

    theta = wrap_angle(
        first_angle + rng.uniform(-0.8, 0.8)
    )

    progress = 0.0
    rows = []

    for step_index in range(MAX_STEPS_PER_EPISODE):

        # Nearest point search along local path
        candidate_s = np.linspace(
            max(0.0, progress - 0.25),
            min(total_length, progress + 1.50),
            75
        )

        candidate_points = np.array([
            interpolate_path(
                planned_path,
                lengths,
                cumulative,
                s
            )
            for s in candidate_s
        ])

        distances = np.linalg.norm(
            candidate_points - position,
            axis=1
        )

        nearest_s = candidate_s[
            np.argmin(distances)
        ]

        progress = max(
            progress,
            float(nearest_s)
        )

        # Same lookahead as STEP 4-1
        lookahead = 0.16

        target_s = min(
            total_length,
            progress + lookahead
        )

        target_point = interpolate_path(
            planned_path,
            lengths,
            cumulative,
            target_s
        )

        delta = target_point - position

        target_distance = np.linalg.norm(delta)

        target_angle = math.atan2(
            delta[1],
            delta[0]
        )

        heading_error = wrap_angle(
            target_angle - theta
        )

        # Same teacher control law as STEP 4-1
        omega = np.clip(
            2.7 * heading_error,
            -max_turn_speed,
            max_turn_speed
        )

        v = max_speed * max(
            0.08,
            1.0 - 0.9 * min(
                abs(heading_error),
                1.3
            )
        )

        goal_distance = np.linalg.norm(
            episode_goal - position
        )

        if goal_distance < 0.7:
            v = min(
                v,
                0.22 + 0.5 * goal_distance
            )

        sensor_values = read_sensors(
            position,
            theta
        )

        # Store state-action pair
        rows.append({
            "episode_id": episode_id,
            "step": step_index,
            "time": step_index * dt,
            "start_x": episode_start[0],
            "start_y": episode_start[1],
            "goal_x": episode_goal[0],
            "goal_y": episode_goal[1],
            "x": position[0],
            "y": position[1],
            "theta": theta,
            "goal_distance": goal_distance,
            "target_distance": target_distance,
            "heading_error": heading_error,
            "sensor_left": sensor_values[0],
            "sensor_front": sensor_values[1],
            "sensor_right": sensor_values[2],
            "v": v,
            "omega": omega
        })

        # Continuous differential-drive update
        next_position = position + dt * v * np.array([
            np.cos(theta),
            np.sin(theta)
        ])

        next_theta = wrap_angle(
            theta + omega * dt
        )

        if not is_free(
            next_position,
            clearance
        ):
            return "BLOCKED", rows, step_index + 1

        position = next_position
        theta = next_theta

        if np.linalg.norm(
            episode_goal - position
        ) < goal_tolerance:

            return (
                "GOAL REACHED",
                rows,
                step_index + 1
            )

    return "TIMEOUT", rows, MAX_STEPS_PER_EPISODE


# ------------------------------------------------------------
# 5. Collect multiple demonstrations
# ------------------------------------------------------------
print("===== STEP 4-2 DATA COLLECTION =====")

for episode_id in range(NUM_EPISODES):

    episode_start, episode_goal = sample_episode()

    # Skip cases with very close start and goal
    if np.linalg.norm(
        episode_goal - episode_start
    ) < 3.0:
        episode_results.append({
            "episode_id": episode_id,
            "status": "TOO_SHORT",
            "steps": 0,
            "samples": 0
        })
        continue

    status, rows, steps = run_teacher_episode(
        episode_id,
        episode_start,
        episode_goal
    )

    # Use successful trajectories as demonstrations
    if status == "GOAL REACHED":
        all_rows.extend(rows)
        samples = len(rows)
    else:
        samples = 0

    episode_results.append({
        "episode_id": episode_id,
        "status": status,
        "steps": steps,
        "samples": samples
    })

    print(
        f"Episode {episode_id+1:02d}/{NUM_EPISODES}: "
        f"{status:13s} "
        f"steps={steps:4d} "
        f"saved={samples:4d}"
    )


# ------------------------------------------------------------
# 6. Save complete dataset and episode summary
# ------------------------------------------------------------
df = pd.DataFrame(all_rows)
results_df = pd.DataFrame(episode_results)

success_count = int(
    (results_df["status"] == "GOAL REACHED").sum()
)

print("\n===== COLLECTION SUMMARY =====")
print("Episodes attempted:", NUM_EPISODES)
print("Successful episodes:", success_count)
print(
    "Success rate:",
    f"{100 * success_count / NUM_EPISODES:.1f}%"
)
print("Total training samples:", len(df))

print("\nEpisode status breakdown:")
print(results_df["status"].value_counts())

results_df.to_csv(
    "step4_2_episode_results.csv",
    index=False
)

if len(df) == 0:
    raise RuntimeError(
        "No successful teacher trajectories were collected."
    )

df.to_csv(
    "step4_2_teacher_dataset.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Split by episode, NOT by individual time step
# ------------------------------------------------------------
# This avoids putting nearly identical consecutive states
# from the same run into both training and validation sets.

successful_ids = df["episode_id"].unique()

rng.shuffle(successful_ids)

if len(successful_ids) < 2:
    print(
        "WARNING: Fewer than two successful episodes. "
        "Train/validation split was not created."
    )

else:
    n_val = max(
        1,
        int(round(len(successful_ids) * 0.20))
    )

    val_ids = successful_ids[:n_val]
    train_ids = successful_ids[n_val:]

    train_df = df[
        df["episode_id"].isin(train_ids)
    ].copy()

    val_df = df[
        df["episode_id"].isin(val_ids)
    ].copy()

    train_df.to_csv(
        "step4_2_train.csv",
        index=False
    )

    val_df.to_csv(
        "step4_2_validation.csv",
        index=False
    )

    print("\n===== DATASET SPLIT =====")
    print("Training episodes:", len(train_ids))
    print("Validation episodes:", len(val_ids))
    print("Training samples:", len(train_df))
    print("Validation samples:", len(val_df))

# ------------------------------------------------------------
# 8. Display dataset preview
# ------------------------------------------------------------
print("\n===== FEATURES FOR PYTORCH =====")
print("Inputs :", feature_columns)
print("Outputs:", action_columns)

display(
    df[
        ["episode_id"]
        + feature_columns
        + action_columns
    ].head(10)
)

print("\nSaved files:")
print("step4_2_teacher_dataset.csv")
print("step4_2_episode_results.csv")

if len(successful_ids) >= 2:
    print("step4_2_train.csv")
    print("step4_2_validation.csv")

## STEP 4-3：PyTorch MLPによる模倣学習

前のSTEPのセルが完了してから実行してください。


In [ ]:
# ============================================================
# STEP 4-3 : PyTorch による模倣学習
# 2-input / 2-output MLP
# ============================================================

import os
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------
SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# 2. Load teacher datasets
# ------------------------------------------------------------
train_file = "step4_2_train.csv"
val_file = "step4_2_validation.csv"

if not os.path.exists(train_file):
    raise FileNotFoundError(train_file)

if not os.path.exists(val_file):
    raise FileNotFoundError(val_file)

train_df = pd.read_csv(train_file)
val_df = pd.read_csv(val_file)

input_columns = [
    "goal_distance",
    "heading_error"
]

output_columns = [
    "v",
    "omega"
]

X_train = train_df[input_columns].to_numpy(
    dtype=np.float32
)

Y_train = train_df[output_columns].to_numpy(
    dtype=np.float32
)

X_val = val_df[input_columns].to_numpy(
    dtype=np.float32
)

Y_val = val_df[output_columns].to_numpy(
    dtype=np.float32
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

# ------------------------------------------------------------
# 3. Normalize inputs and outputs
#    Statistics are calculated ONLY from training data.
# ------------------------------------------------------------
x_mean = X_train.mean(axis=0)
x_std = X_train.std(axis=0)

y_mean = Y_train.mean(axis=0)
y_std = Y_train.std(axis=0)

x_std = np.maximum(x_std, 1e-6)
y_std = np.maximum(y_std, 1e-6)

X_train_n = (X_train - x_mean) / x_std
X_val_n = (X_val - x_mean) / x_std

Y_train_n = (Y_train - y_mean) / y_std
Y_val_n = (Y_val - y_mean) / y_std

print("\nInput mean:", x_mean)
print("Input std :", x_std)
print("Output mean:", y_mean)
print("Output std :", y_std)

# ------------------------------------------------------------
# 4. PyTorch datasets
# ------------------------------------------------------------
train_dataset = TensorDataset(
    torch.from_numpy(X_train_n),
    torch.from_numpy(Y_train_n)
)

val_dataset = TensorDataset(
    torch.from_numpy(X_val_n),
    torch.from_numpy(Y_val_n)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False
)

# ------------------------------------------------------------
# 5. Define MLP
# ------------------------------------------------------------
class ImitationMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.network(x)


model = ImitationMLP().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("\nModel architecture:")
print(model)

# ------------------------------------------------------------
# 6. Training
# ------------------------------------------------------------
MAX_EPOCHS = 150
PATIENCE = 15

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_state = None
best_epoch = 0
wait = 0

for epoch in range(MAX_EPOCHS):

    # Training phase
    model.train()

    train_loss_sum = 0.0
    train_count = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()

        prediction = model(xb)

        loss = criterion(prediction, yb)

        loss.backward()
        optimizer.step()

        batch_size = len(xb)

        train_loss_sum += loss.item() * batch_size
        train_count += batch_size

    epoch_train_loss = (
        train_loss_sum / train_count
    )

    # Validation phase
    model.eval()

    val_loss_sum = 0.0
    val_count = 0

    with torch.no_grad():

        for xb, yb in val_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            prediction = model(xb)

            loss = criterion(prediction, yb)

            batch_size = len(xb)

            val_loss_sum += loss.item() * batch_size
            val_count += batch_size

    epoch_val_loss = val_loss_sum / val_count

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    # Save best model in memory
    if epoch_val_loss < best_val_loss - 1e-7:

        best_val_loss = epoch_val_loss
        best_epoch = epoch + 1

        best_state = copy.deepcopy(
            model.state_dict()
        )

        wait = 0

    else:
        wait += 1

    if (
        epoch == 0
        or (epoch + 1) % 10 == 0
    ):
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train MSE: {epoch_train_loss:.6f} | "
            f"Val MSE: {epoch_val_loss:.6f}"
        )

    if wait >= PATIENCE:

        print(
            f"\nEarly stopping at epoch {epoch+1}"
        )

        break

# Restore best model
model.load_state_dict(best_state)
model.eval()

print("\n===== TRAINING RESULT =====")
print("Epochs executed:", len(train_losses))
print("Best epoch:", best_epoch)
print("Best validation MSE:", best_val_loss)

# ------------------------------------------------------------
# 7. Plot learning curves
# ------------------------------------------------------------
epochs = np.arange(1, len(train_losses) + 1)

plt.figure(figsize=(9, 5))

plt.plot(
    epochs,
    train_losses,
    label="Training loss",
    linewidth=2
)

plt.plot(
    epochs,
    val_losses,
    label="Validation loss",
    linewidth=2
)

plt.axvline(
    best_epoch,
    color="gray",
    linestyle="--",
    alpha=0.7,
    label=f"Best epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Normalized MSE")
plt.title("STEP 4-3: Imitation Learning Curves")

plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plt.savefig(
    "step4_3_learning_curves.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 8. Evaluate prediction accuracy
#    Convert predictions back to physical units.
# ------------------------------------------------------------
model.eval()

with torch.no_grad():

    val_predictions_n = model(
        torch.from_numpy(X_val_n).to(device)
    ).cpu().numpy()

val_predictions = (
    val_predictions_n * y_std + y_mean
)

mae = np.mean(
    np.abs(val_predictions - Y_val),
    axis=0
)

rmse = np.sqrt(
    np.mean(
        (val_predictions - Y_val) ** 2,
        axis=0
    )
)

print("\n===== VALIDATION ACCURACY =====")

print(
    f"Forward speed v: "
    f"MAE={mae[0]:.6f} m/s, "
    f"RMSE={rmse[0]:.6f} m/s"
)

print(
    f"Angular speed omega: "
    f"MAE={mae[1]:.6f} rad/s, "
    f"RMSE={rmse[1]:.6f} rad/s"
)

# ------------------------------------------------------------
# 9. Compare teacher vs AI predictions
# ------------------------------------------------------------
fig, axes = plt.subplots(
    1, 2,
    figsize=(11, 4.5)
)

names = [
    "Forward speed v (m/s)",
    "Angular speed omega (rad/s)"
]

for i, ax in enumerate(axes):

    ax.scatter(
        Y_val[:, i],
        val_predictions[:, i],
        s=4,
        alpha=0.2
    )

    lower = min(
        Y_val[:, i].min(),
        val_predictions[:, i].min()
    )

    upper = max(
        Y_val[:, i].max(),
        val_predictions[:, i].max()
    )

    ax.plot(
        [lower, upper],
        [lower, upper],
        "k--",
        linewidth=1
    )

    ax.set_xlabel("Teacher")
    ax.set_ylabel("AI prediction")
    ax.set_title(names[i])
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "STEP 4-3: Teacher vs AI Predictions"
)

plt.tight_layout()

plt.savefig(
    "step4_3_prediction_comparison.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 10. Save trained model and normalization parameters
# ------------------------------------------------------------
model_file = "step4_3_imitation_model.pth"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "x_mean": x_mean,
        "x_std": x_std,
        "y_mean": y_mean,
        "y_std": y_std,
        "input_columns": input_columns,
        "output_columns": output_columns,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "seed": SEED
    },
    model_file
)

history_df = pd.DataFrame({
    "epoch": epochs,
    "train_loss": train_losses,
    "val_loss": val_losses
})

history_df.to_csv(
    "step4_3_training_history.csv",
    index=False
)

print("\n===== SAVED FILES =====")
print(model_file)
print("step4_3_learning_curves.png")
print("step4_3_prediction_comparison.png")
print("step4_3_training_history.csv")

print("\nSTEP 4-3 COMPLETED")


## STEP 4-4：基準環境での教師・AI閉ループ比較

前のSTEPのセルが完了してから実行してください。


In [ ]:
# ============================================================
# STEP 4-4 : 教師とAIの走行比較コード
# Closed-loop robot navigation comparison
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn

from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import Video, display

# ------------------------------------------------------------
# 1. Experiment settings
# ------------------------------------------------------------
EXP_START = np.array([1.0, 1.0], dtype=float)
EXP_GOAL = np.array([9.0, 9.0], dtype=float)

EXP_DT = 0.05
EXP_FPS = 20
EXP_MAX_STEPS = 2000

EXP_MAX_V = 0.40
EXP_MAX_OMEGA = 1.20
EXP_GOAL_TOL = 0.35
EXP_CLEARANCE = 0.55

MODEL_FILE = "step4_3_imitation_model.pth"

if not os.path.exists(MODEL_FILE):
    raise FileNotFoundError(
        f"Model not found: {MODEL_FILE}"
    )

# ------------------------------------------------------------
# 2. Load trained MLP
# ------------------------------------------------------------
class Step44MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.network(x)


# This file was created by our own STEP 4-3 code.
# Only use weights_only=False for trusted checkpoints.
checkpoint = torch.load(
    MODEL_FILE,
    map_location="cpu",
    weights_only=False
)

ai_model = Step44MLP()
ai_model.load_state_dict(
    checkpoint["model_state_dict"]
)
ai_model.eval()

x_mean_44 = np.asarray(
    checkpoint["x_mean"], dtype=np.float32
)
x_std_44 = np.asarray(
    checkpoint["x_std"], dtype=np.float32
)
y_mean_44 = np.asarray(
    checkpoint["y_mean"], dtype=np.float32
)
y_std_44 = np.asarray(
    checkpoint["y_std"], dtype=np.float32
)

assert checkpoint["input_columns"] == [
    "goal_distance", "heading_error"
]

assert checkpoint["output_columns"] == [
    "v", "omega"
]

print("AI model loaded:", MODEL_FILE)
print("Best training epoch:", checkpoint["best_epoch"])

# ------------------------------------------------------------
# 3. A* route for both controllers
# ------------------------------------------------------------
planned_path_44 = astar(
    EXP_START,
    EXP_GOAL
)

path_lengths_44 = np.linalg.norm(
    np.diff(planned_path_44, axis=0),
    axis=1
)

path_cumulative_44 = np.concatenate(
    [[0.0], np.cumsum(path_lengths_44)]
)

path_total_44 = path_cumulative_44[-1]

print("A* path points:", len(planned_path_44))


def path_point_44(s):

    s = float(np.clip(s, 0, path_total_44))

    index = np.searchsorted(
        path_cumulative_44,
        s,
        side="right"
    ) - 1

    index = min(
        index,
        len(path_lengths_44) - 1
    )

    length = path_lengths_44[index]

    if length < 1e-10:
        return planned_path_44[index].copy()

    fraction = (
        s - path_cumulative_44[index]
    ) / length

    return (
        planned_path_44[index]
        + fraction * (
            planned_path_44[index + 1]
            - planned_path_44[index]
        )
    )


# ------------------------------------------------------------
# 4. Teacher and AI control laws
# ------------------------------------------------------------
def teacher_control_44(
    goal_distance,
    heading_error
):

    omega = np.clip(
        2.7 * heading_error,
        -EXP_MAX_OMEGA,
        EXP_MAX_OMEGA
    )

    v = EXP_MAX_V * max(
        0.08,
        1.0 - 0.9 * min(
            abs(heading_error),
            1.3
        )
    )

    if goal_distance < 0.7:
        v = min(
            v,
            0.22 + 0.5 * goal_distance
        )

    return float(v), float(omega)


def ai_control_44(
    goal_distance,
    heading_error
):

    state = np.array(
        [goal_distance, heading_error],
        dtype=np.float32
    )

    state_n = (
        state - x_mean_44
    ) / x_std_44

    with torch.no_grad():

        prediction_n = ai_model(
            torch.from_numpy(
                state_n.reshape(1, 2)
            )
        ).numpy()[0]

    prediction = (
        prediction_n * y_std_44 + y_mean_44
    )

    # Physical actuator limits
    v = float(np.clip(
        prediction[0],
        0.0,
        EXP_MAX_V
    ))

    omega = float(np.clip(
        prediction[1],
        -EXP_MAX_OMEGA,
        EXP_MAX_OMEGA
    ))

    return v, omega


# ------------------------------------------------------------
# 5. Distance to physical obstacles
# ------------------------------------------------------------
def obstacle_clearance_44(position):

    x, y = position

    # Distance to world boundary
    boundary_distance = min(
        x, y,
        world_size - x,
        world_size - y
    )

    minimum = boundary_distance

    for cx, cy, w, h in obstacles:

        dx = max(
            abs(x - cx) - w / 2,
            0.0
        )

        dy = max(
            abs(y - cy) - h / 2,
            0.0
        )

        distance = math.hypot(dx, dy)

        minimum = min(
            minimum,
            distance
        )

    return float(minimum)


# ------------------------------------------------------------
# 6. Run one independent closed-loop simulation
# ------------------------------------------------------------
def simulate_44(controller_name):

    position = EXP_START.copy()
    theta = 0.0
    progress = 0.0

    positions = [position.copy()]
    headings = [theta]

    records = []
    status = "TIMEOUT"

    for step in range(EXP_MAX_STEPS):

        # Same route tracking logic as STEP 4-1
        candidate_s = np.linspace(
            max(0.0, progress - 0.25),
            min(
                path_total_44,
                progress + 1.50
            ),
            75
        )

        candidate_points = np.array([
            path_point_44(s)
            for s in candidate_s
        ])

        distances = np.linalg.norm(
            candidate_points - position,
            axis=1
        )

        nearest_s = candidate_s[
            np.argmin(distances)
        ]

        progress = max(
            progress,
            float(nearest_s)
        )

        target_s = min(
            path_total_44,
            progress + 0.16
        )

        target = path_point_44(target_s)

        delta = target - position

        target_angle = math.atan2(
            delta[1],
            delta[0]
        )

        heading_error = wrap_angle(
            target_angle - theta
        )

        goal_distance = float(
            np.linalg.norm(
                EXP_GOAL - position
            )
        )

        # Controller selection
        if controller_name == "Teacher":

            v, omega = teacher_control_44(
                goal_distance,
                heading_error
            )

        elif controller_name == "AI":

            v, omega = ai_control_44(
                goal_distance,
                heading_error
            )

        else:
            raise ValueError(controller_name)

        # Save state and action
        records.append({
            "controller": controller_name,
            "time": step * EXP_DT,
            "x": position[0],
            "y": position[1],
            "theta": theta,
            "goal_distance": goal_distance,
            "heading_error": heading_error,
            "v": v,
            "omega": omega,
            "obstacle_distance":
                obstacle_clearance_44(position)
        })

        # Differential-drive kinematics
        next_position = (
            position
            + EXP_DT * v * np.array([
                np.cos(theta),
                np.sin(theta)
            ])
        )

        next_theta = wrap_angle(
            theta + omega * EXP_DT
        )

        if not is_free(
            next_position,
            EXP_CLEARANCE
        ):
            status = "BLOCKED"
            break

        position = next_position
        theta = next_theta

        positions.append(position.copy())
        headings.append(theta)

        if np.linalg.norm(
            EXP_GOAL - position
        ) < EXP_GOAL_TOL:

            status = "GOAL REACHED"
            break

    return {
        "name": controller_name,
        "status": status,
        "positions": np.array(positions),
        "headings": np.array(headings),
        "records": pd.DataFrame(records),
        "time": (
            len(positions) - 1
        ) * EXP_DT,
        "final_position": position.copy()
    }


# ------------------------------------------------------------
# 7. Run both controllers
# ------------------------------------------------------------
teacher_result = simulate_44("Teacher")
ai_result = simulate_44("AI")

print("\n===== CLOSED-LOOP RESULTS =====")

for result in [
    teacher_result,
    ai_result
]:

    print(
        f"{result['name']:8s} | "
        f"{result['status']:12s} | "
        f"Time: {result['time']:.2f} s | "
        f"Final: {result['final_position']}"
    )


# ------------------------------------------------------------
# 8. Performance metrics
# ------------------------------------------------------------
def calculate_metrics_44(result):

    positions = result["positions"]
    data = result["records"]

    travel_distance = float(
        np.sum(
            np.linalg.norm(
                np.diff(positions, axis=0),
                axis=1
            )
        )
    )

    minimum_obstacle_distance = min(
        obstacle_clearance_44(p)
        for p in positions
    )

    omega_values = data["omega"].to_numpy()

    if len(omega_values) >= 2:

        # RMS angular acceleration
        angular_acceleration = (
            np.diff(omega_values) / EXP_DT
        )

        angular_smoothness = float(
            np.sqrt(
                np.mean(
                    angular_acceleration ** 2
                )
            )
        )

    else:
        angular_smoothness = np.nan

    return {
        "Controller": result["name"],
        "Status": result["status"],
        "Time (s)": result["time"],
        "Path length (m)": travel_distance,
        "Min obstacle distance (m)":
            minimum_obstacle_distance,
        "Min safety margin (m)":
            minimum_obstacle_distance
            - EXP_CLEARANCE,
        "Angular accel RMS (rad/s^2)":
            angular_smoothness
    }


comparison_df = pd.DataFrame([
    calculate_metrics_44(teacher_result),
    calculate_metrics_44(ai_result)
])

print("\n===== PERFORMANCE COMPARISON =====")
display(comparison_df.round(5))

comparison_df.to_csv(
    "step4_4_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# 9. Static trajectory comparison
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 8))

ax.set_xlim(0, world_size)
ax.set_ylim(0, world_size)
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)

for cx, cy, w, h in obstacles:

    ax.add_patch(
        patches.Rectangle(
            (cx - w/2, cy - h/2),
            w, h,
            facecolor="salmon",
            edgecolor="darkred",
            alpha=0.65
        )
    )

ax.plot(
    planned_path_44[:, 0],
    planned_path_44[:, 1],
    "--",
    color="gray",
    alpha=0.5,
    label="A* path"
)

ax.plot(
    teacher_result["positions"][:, 0],
    teacher_result["positions"][:, 1],
    color="royalblue",
    linewidth=2.5,
    label="Teacher"
)

ax.plot(
    ai_result["positions"][:, 0],
    ai_result["positions"][:, 1],
    color="darkorange",
    linewidth=2,
    linestyle="--",
    label="AI"
)

ax.plot(
    EXP_START[0],
    EXP_START[1],
    "go",
    markersize=9,
    label="Start"
)

ax.plot(
    EXP_GOAL[0],
    EXP_GOAL[1],
    "r*",
    markersize=14,
    label="Goal"
)

ax.set_title(
    "STEP 4-4: Teacher vs AI Trajectories"
)

ax.legend(loc="lower right")
plt.tight_layout()

plt.savefig(
    "step4_4_trajectory_comparison.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 10. Control signal comparison
# ------------------------------------------------------------
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 7),
    sharex=True
)

for result, color in [
    (teacher_result, "royalblue"),
    (ai_result, "darkorange")
]:

    data = result["records"]

    axes[0].plot(
        data["time"],
        data["v"],
        color=color,
        label=result["name"],
        alpha=0.85
    )

    axes[1].plot(
        data["time"],
        data["omega"],
        color=color,
        label=result["name"],
        alpha=0.85
    )

axes[0].set_ylabel("Forward speed v (m/s)")
axes[1].set_ylabel("Angular speed omega (rad/s)")
axes[1].set_xlabel("Time (s)")

for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.suptitle(
    "STEP 4-4: Control Signals"
)

plt.tight_layout()

plt.savefig(
    "step4_4_control_comparison.png",
    dpi=180
)

plt.show()

# ------------------------------------------------------------
# 11. Side-by-side animation
# ------------------------------------------------------------
# Use every 2nd simulation frame to keep video compact.
# Playback duration is reduced by half, but time labels
# indicate the actual simulation time.

FRAME_STRIDE = 2

teacher_pos = teacher_result["positions"]
ai_pos = ai_result["positions"]

max_frames = max(
    len(teacher_pos),
    len(ai_pos)
)

animation_frames = list(
    range(0, max_frames, FRAME_STRIDE)
)

if animation_frames[-1] != max_frames - 1:
    animation_frames.append(max_frames - 1)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 6)
)

robot_artists = []
trail_artists = []
heading_artists = []
time_artists = []

for ax, result, color in zip(
    axes,
    [teacher_result, ai_result],
    ["royalblue", "darkorange"]
):

    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

    for cx, cy, w, h in obstacles:

        ax.add_patch(
            patches.Rectangle(
                (cx-w/2, cy-h/2),
                w, h,
                facecolor="salmon",
                edgecolor="darkred",
                alpha=0.65
            )
        )

    ax.plot(
        planned_path_44[:, 0],
        planned_path_44[:, 1],
        "--",
        color="gray",
        alpha=0.45,
        linewidth=1
    )

    ax.plot(
        EXP_START[0],
        EXP_START[1],
        "go"
    )

    ax.plot(
        EXP_GOAL[0],
        EXP_GOAL[1],
        "r*",
        markersize=12
    )

    ax.set_title(
        f"{result['name']}: {result['status']}"
    )

    trail, = ax.plot(
        [], [],
        color=color,
        linewidth=2
    )

    robot = patches.Circle(
        EXP_START,
        robot_radius,
        facecolor=color,
        edgecolor="black",
        alpha=0.85
    )

    ax.add_patch(robot)

    heading, = ax.plot(
        [], [],
        color="black",
        linewidth=2
    )

    time_label = ax.text(
        0.03, 0.97, "",
        transform=ax.transAxes,
        va="top"
    )

    trail_artists.append(trail)
    robot_artists.append(robot)
    heading_artists.append(heading)
    time_artists.append(time_label)


def update_44(frame_number):

    simulation_index = animation_frames[
        frame_number
    ]

    artists = []

    for i, result in enumerate(
        [teacher_result, ai_result]
    ):

        positions = result["positions"]
        headings = result["headings"]

        index = min(
            simulation_index,
            len(positions) - 1
        )

        p = positions[index]
        theta = headings[index]

        trail_artists[i].set_data(
            positions[:index+1, 0],
            positions[:index+1, 1]
        )

        robot_artists[i].center = p

        tip = p + 0.38 * np.array([
            np.cos(theta),
            np.sin(theta)
        ])

        heading_artists[i].set_data(
            [p[0], tip[0]],
            [p[1], tip[1]]
        )

        time_artists[i].set_text(
            f"Simulation time: "
            f"{index * EXP_DT:.1f} s"
        )

        artists.extend([
            trail_artists[i],
            robot_artists[i],
            heading_artists[i],
            time_artists[i]
        ])

    return artists


animation = FuncAnimation(
    fig,
    update_44,
    frames=len(animation_frames),
    interval=1000 / EXP_FPS,
    blit=True
)

video_filename = (
    "step4_4_teacher_vs_ai.mp4"
)

animation.save(
    video_filename,
    writer=FFMpegWriter(
        fps=EXP_FPS,
        bitrate=1800
    )
)

plt.close(fig)

display(
    Video(
        video_filename,
        embed=True,
        width=900
    )
)

print("\n===== STEP 4-4 COMPLETED =====")
print("Teacher:", teacher_result["status"])
print("AI:", ai_result["status"])
print("Video:", video_filename)
print("Comparison:", "step4_4_comparison.csv")
print("Trajectory:", "step4_4_trajectory_comparison.png")
print("Control:", "step4_4_control_comparison.png")

## STEP 4-5 ver.2：異なるスタート・障害物配置での汎化実験

前のSTEPのセルが完了してから実行してください。


In [ ]:
# ============================================================
# STEP 4-5 : 汎化実験と比較動画の作成
#
# Experiment 1: New start position and heading
# Experiment 2: Modified obstacle arrangement
#
# Teacher vs Frozen Imitation Learning AI
# Side-by-side MP4 videos + metrics + trajectory plots
#
# Run AFTER STEP 4-4 in the same Colab runtime.
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import Video, display

# ------------------------------------------------------------
# 1. Check prerequisites
# ------------------------------------------------------------
required_names = [
    "astar",
    "is_free",
    "wrap_angle",
    "teacher_control_44",
    "ai_control_44",
    "ai_model",
    "obstacles",
    "world_size",
    "robot_radius"
]

missing = [
    name for name in required_names
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Please run STEP 4-1 and STEP 4-4 first. "
        f"Missing: {missing}"
    )

# Confirm that the model remains frozen
ai_model.eval()

for parameter in ai_model.parameters():
    parameter.requires_grad_(False)

print("Frozen AI model ready.")
print("No additional training will be performed.")

# ------------------------------------------------------------
# 2. Fixed experiment parameters
# ------------------------------------------------------------
DT_45 = 0.05
FPS_45 = 20
MAX_STEPS_45 = 2200

GOAL_TOL_45 = 0.35
CLEARANCE_45 = 0.55

ORIGINAL_OBSTACLES_45 = [
    tuple(item) for item in obstacles
]

# Experiment 1:
# Same obstacles, different start and initial heading.
#
# Experiment 2:
# Different obstacle locations.
#
# These conditions are not included in STEP 4-2
# teacher demonstrations.

experiments_45 = [
    {
        "id": "experiment_1",
        "title": "Opposite-Side Start",
        "start": (8.8, 1.2),
        "goal": (9.0, 9.0),
        "theta": 2.8,
        "obstacles": ORIGINAL_OBSTACLES_45
    },
    {
        "id": "experiment_2",
        "title": "Completely Redesigned Map",
        "start": (1.0, 1.0),
        "goal": (9.0, 9.0),
        "theta": 1.0,
        "obstacles": [
            (3.0, 3.0, 2.0, 0.8),
            (7.0, 3.0, 2.0, 0.8),
            (3.0, 6.0, 2.0, 0.8),
            (7.0, 7.0, 2.0, 0.8)
        ]
    }
]

# ------------------------------------------------------------
# STEP 4-5 ver.2:
# Obstacle-aware A* and collision checking
# ------------------------------------------------------------
import heapq

def is_free(position, clearance=0.55):

    x, y = np.asarray(position, dtype=float)

    if min(x, y, world_size-x, world_size-y) < clearance:
        return False

    for cx, cy, w, h in obstacles:

        dx = max(abs(x-cx)-w/2, 0.0)
        dy = max(abs(y-cy)-h/2, 0.0)

        if math.hypot(dx, dy) < clearance:
            return False

    return True


def astar(start, goal):

    resolution = 0.20
    planning_clearance = 0.57

    start = np.asarray(start, dtype=float)
    goal = np.asarray(goal, dtype=float)

    if not is_free(start, planning_clearance):
        raise RuntimeError("Start is not collision-free")

    if not is_free(goal, planning_clearance):
        raise RuntimeError("Goal is not collision-free")

    def to_grid(p):
        return tuple(
            int(round(v/resolution)) for v in p
        )

    def to_world(p):
        return np.array(p, dtype=float) * resolution

    start_node = to_grid(start)
    goal_node = to_grid(goal)

    def valid(node):
        return is_free(
            to_world(node),
            planning_clearance
        )

    moves = [
        (1,0), (-1,0), (0,1), (0,-1),
        (1,1), (1,-1), (-1,1), (-1,-1)
    ]

    open_heap = [
        (0.0, start_node)
    ]

    came_from = {}
    g_score = {start_node: 0.0}
    closed = set()

    while open_heap:

        _, current = heapq.heappop(open_heap)

        if current in closed:
            continue

        closed.add(current)

        if current == goal_node:

            nodes = [current]

            while current in came_from:
                current = came_from[current]
                nodes.append(current)

            nodes.reverse()

            path = np.array(
                [to_world(p) for p in nodes]
            )

            # Preserve exact endpoints
            path[0] = start
            path[-1] = goal

            return path

        for dx, dy in moves:

            neighbor = (
                current[0] + dx,
                current[1] + dy
            )

            if not valid(neighbor):
                continue

            # Prevent cutting diagonally through
            # obstacle corners
            if dx != 0 and dy != 0:

                side_a = (
                    current[0] + dx,
                    current[1]
                )

                side_b = (
                    current[0],
                    current[1] + dy
                )

                if not (
                    valid(side_a) and valid(side_b)
                ):
                    continue

            tentative_g = (
                g_score[current]
                + math.hypot(dx, dy)
            )

            if tentative_g < g_score.get(
                neighbor, float("inf")
            ):

                came_from[neighbor] = current
                g_score[neighbor] = tentative_g

                heuristic = math.hypot(
                    neighbor[0] - goal_node[0],
                    neighbor[1] - goal_node[1]
                )

                heapq.heappush(
                    open_heap,
                    (
                        tentative_g + heuristic,
                        neighbor
                    )
                )

    raise RuntimeError("No collision-free A* path")

# ------------------------------------------------------------
# 3. Local path geometry
# ------------------------------------------------------------
def make_path_45(start, goal):

    planned = astar(start, goal)

    if len(planned) < 2:
        raise RuntimeError("A* path is too short.")

    lengths = np.linalg.norm(
        np.diff(planned, axis=0),
        axis=1
    )

    cumulative = np.concatenate(
        [[0.0], np.cumsum(lengths)]
    )

    return {
        "points": planned,
        "lengths": lengths,
        "cumulative": cumulative,
        "total": float(cumulative[-1])
    }


def path_point_45(path_info, s):

    points = path_info["points"]
    lengths = path_info["lengths"]
    cumulative = path_info["cumulative"]
    total = path_info["total"]

    s = float(np.clip(s, 0.0, total))

    index = np.searchsorted(
        cumulative,
        s,
        side="right"
    ) - 1

    index = min(index, len(lengths) - 1)

    length = lengths[index]

    if length < 1e-10:
        return points[index].copy()

    fraction = (
        s - cumulative[index]
    ) / length

    return (
        points[index]
        + fraction * (
            points[index + 1] - points[index]
        )
    )


# ------------------------------------------------------------
# 4. Physical distance to obstacles and walls
# ------------------------------------------------------------
def clearance_distance_45(position):

    x, y = position

    minimum = min(
        x,
        y,
        world_size - x,
        world_size - y
    )

    for cx, cy, w, h in obstacles:

        dx = max(
            abs(x - cx) - w / 2,
            0.0
        )

        dy = max(
            abs(y - cy) - h / 2,
            0.0
        )

        minimum = min(
            minimum,
            math.hypot(dx, dy)
        )

    return float(minimum)


# ------------------------------------------------------------
# 5. Independent closed-loop simulation
# ------------------------------------------------------------
def simulate_45(
    controller_name,
    start,
    goal,
    initial_theta,
    path_info
):

    position = np.array(
        start,
        dtype=float
    )

    goal = np.array(
        goal,
        dtype=float
    )

    theta = float(initial_theta)
    progress = 0.0

    positions = [position.copy()]
    headings = [theta]
    records = []

    status = "TIMEOUT"

    if not is_free(position, CLEARANCE_45):
        return {
            "name": controller_name,
            "status": "INVALID_START",
            "positions": np.array(positions),
            "headings": np.array(headings),
            "records": pd.DataFrame(),
            "time": 0.0,
            "final_position": position.copy()
        }

    for step in range(MAX_STEPS_45):

        # Same tracking method as STEP 4-1 / 4-4
        candidate_s = np.linspace(
            max(0.0, progress - 0.25),
            min(
                path_info["total"],
                progress + 1.50
            ),
            75
        )

        candidate_points = np.array([
            path_point_45(path_info, s)
            for s in candidate_s
        ])

        distances = np.linalg.norm(
            candidate_points - position,
            axis=1
        )

        nearest_s = candidate_s[
            np.argmin(distances)
        ]

        progress = max(
            progress,
            float(nearest_s)
        )

        target_s = min(
            path_info["total"],
            progress + 0.16
        )

        target = path_point_45(
            path_info,
            target_s
        )

        delta = target - position

        target_angle = math.atan2(
            delta[1],
            delta[0]
        )

        heading_error = wrap_angle(
            target_angle - theta
        )

        goal_distance = float(
            np.linalg.norm(goal - position)
        )

        # Controller selection
        if controller_name == "Teacher":

            v, omega = teacher_control_44(
                goal_distance,
                heading_error
            )

        elif controller_name == "AI":

            v, omega = ai_control_44(
                goal_distance,
                heading_error
            )

        else:
            raise ValueError(controller_name)

        obstacle_distance = (
            clearance_distance_45(position)
        )

        records.append({
            "step": step,
            "time": step * DT_45,
            "x": position[0],
            "y": position[1],
            "theta": theta,
            "goal_distance": goal_distance,
            "heading_error": heading_error,
            "v": v,
            "omega": omega,
            "obstacle_distance": obstacle_distance
        })

        # Differential-drive kinematics
        next_position = (
            position
            + DT_45 * v * np.array([
                np.cos(theta),
                np.sin(theta)
            ])
        )

        next_theta = wrap_angle(
            theta + omega * DT_45
        )

        if not is_free(
            next_position,
            CLEARANCE_45
        ):
            status = "BLOCKED"
            break

        position = next_position
        theta = next_theta

        positions.append(position.copy())
        headings.append(theta)

        if np.linalg.norm(
            goal - position
        ) < GOAL_TOL_45:

            status = "GOAL REACHED"
            break

    return {
        "name": controller_name,
        "status": status,
        "positions": np.array(positions),
        "headings": np.array(headings),
        "records": pd.DataFrame(records),
        "time": (
            len(positions) - 1
        ) * DT_45,
        "final_position": position.copy()
    }


# ------------------------------------------------------------
# 6. Calculate performance metrics
# ------------------------------------------------------------
def metrics_45(result, experiment_id):

    positions = result["positions"]
    data = result["records"]

    path_length = float(
        np.sum(
            np.linalg.norm(
                np.diff(positions, axis=0),
                axis=1
            )
        )
    )

    minimum_distance = min(
        clearance_distance_45(p)
        for p in positions
    )

    if len(data) >= 2:

        omega = data["omega"].to_numpy()

        angular_accel = (
            np.diff(omega) / DT_45
        )

        smoothness = float(
            np.sqrt(
                np.mean(angular_accel ** 2)
            )
        )

    else:
        smoothness = np.nan

    return {
        "Experiment": experiment_id,
        "Controller": result["name"],
        "Status": result["status"],
        "Time (s)": result["time"],
        "Path length (m)": path_length,
        "Min obstacle distance (m)":
            minimum_distance,
        "Min safety margin (m)":
            minimum_distance - CLEARANCE_45,
        "Angular accel RMS (rad/s^2)":
            smoothness
    }


# ------------------------------------------------------------
# 7. Draw obstacle map
# ------------------------------------------------------------
def draw_map_45(ax, experiment, path_info):

    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

    for cx, cy, w, h in experiment["obstacles"]:

        ax.add_patch(
            patches.Rectangle(
                (cx - w/2, cy - h/2),
                w,
                h,
                facecolor="salmon",
                edgecolor="darkred",
                alpha=0.65
            )
        )

    points = path_info["points"]

    ax.plot(
        points[:, 0],
        points[:, 1],
        "--",
        color="gray",
        linewidth=1,
        alpha=0.5
    )

    sx, sy = experiment["start"]
    gx, gy = experiment["goal"]

    ax.plot(
        sx, sy,
        "go",
        markersize=8
    )

    ax.plot(
        gx, gy,
        "r*",
        markersize=13
    )


# ------------------------------------------------------------
# 8. Static trajectory comparison
# ------------------------------------------------------------
def save_trajectory_plot_45(
    experiment,
    path_info,
    teacher_result,
    ai_result
):

    fig, ax = plt.subplots(figsize=(8, 8))

    draw_map_45(
        ax,
        experiment,
        path_info
    )

    for result, color, style in [
        (teacher_result, "royalblue", "-"),
        (ai_result, "darkorange", "--")
    ]:

        positions = result["positions"]

        ax.plot(
            positions[:, 0],
            positions[:, 1],
            color=color,
            linestyle=style,
            linewidth=2.5,
            label=(
                f"{result['name']} "
                f"({result['status']})"
            )
        )

    ax.set_title(
        f"STEP 4-5: {experiment['title']}"
    )

    ax.legend(loc="lower right")

    plt.tight_layout()

    filename = (
        f"step4_5_{experiment['id']}_trajectory.png"
    )

    plt.savefig(filename, dpi=180)
    plt.show()

    return filename


# ------------------------------------------------------------
# 9. Side-by-side MP4 comparison
# ------------------------------------------------------------
def save_comparison_video_45(
    experiment,
    path_info,
    teacher_result,
    ai_result
):

    results = [
        teacher_result,
        ai_result
    ]

    colors = [
        "royalblue",
        "darkorange"
    ]

    fig, axes = plt.subplots(
        1, 2,
        figsize=(12, 6)
    )

    robots = []
    trails = []
    headings_art = []
    labels = []

    for ax, result, color in zip(
        axes,
        results,
        colors
    ):

        draw_map_45(
            ax,
            experiment,
            path_info
        )

        ax.set_title(
            f"{result['name']}: "
            f"{result['status']}"
        )

        trail, = ax.plot(
            [], [],
            color=color,
            linewidth=2.5
        )

        robot = patches.Circle(
            experiment["start"],
            robot_radius,
            facecolor=color,
            edgecolor="black",
            alpha=0.85
        )

        ax.add_patch(robot)

        heading, = ax.plot(
            [], [],
            color="black",
            linewidth=2
        )

        label = ax.text(
            0.03,
            0.97,
            "",
            transform=ax.transAxes,
            va="top",
            fontsize=10
        )

        trails.append(trail)
        robots.append(robot)
        headings_art.append(heading)
        labels.append(label)

    max_length = max(
        len(teacher_result["positions"]),
        len(ai_result["positions"])
    )

    # 2 simulation steps per animation frame.
    # This makes the video shorter without changing
    # the underlying simulation or performance metrics.
    stride = 2

    frames = list(
        range(0, max_length, stride)
    )

    if frames[-1] != max_length - 1:
        frames.append(max_length - 1)

    def update(frame_index):

        simulation_step = frames[frame_index]

        artists = []

        for i, result in enumerate(results):

            positions = result["positions"]
            angles = result["headings"]

            index = min(
                simulation_step,
                len(positions) - 1
            )

            p = positions[index]
            theta = angles[index]

            trails[i].set_data(
                positions[:index+1, 0],
                positions[:index+1, 1]
            )

            robots[i].center = p

            tip = p + 0.38 * np.array([
                np.cos(theta),
                np.sin(theta)
            ])

            headings_art[i].set_data(
                [p[0], tip[0]],
                [p[1], tip[1]]
            )

            labels[i].set_text(
                f"Simulation time: "
                f"{index * DT_45:.1f} s"
            )

            artists.extend([
                trails[i],
                robots[i],
                headings_art[i],
                labels[i]
            ])

        return artists

    animation = FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / FPS_45,
        blit=True
    )

    filename = (
        f"step4_5_{experiment['id']}_comparison.mp4"
    )

    animation.save(
        filename,
        writer=FFMpegWriter(
            fps=FPS_45,
            bitrate=1800
        )
    )

    plt.close(fig)

    print("Video saved:", filename)

    display(
        Video(
            filename,
            embed=True,
            width=900
        )
    )

    return filename


# ------------------------------------------------------------
# 10. Run both generalization experiments
# ------------------------------------------------------------
all_metrics_45 = []
all_results_45 = {}

try:

    for experiment in experiments_45:

        experiment_id = experiment["id"]

        print("\n" + "=" * 55)
        print("STEP 4-5:", experiment["title"])
        print("=" * 55)

        # Apply experimental obstacle arrangement.
        # Existing A* and collision functions read this
        # global variable when they are called.
        obstacles = [
            tuple(item)
            for item in experiment["obstacles"]
        ]

        start = np.array(
            experiment["start"],
            dtype=float
        )

        goal = np.array(
            experiment["goal"],
            dtype=float
        )

        theta = float(
            experiment["theta"]
        )

        print("Start:", start)
        print("Goal:", goal)
        print("Initial heading:", theta)
        print("Obstacle count:", len(obstacles))

        # Replan using A* for this environment
        try:
            path_info = make_path_45(
                start,
                goal
            )

        except RuntimeError as error:
            print("A* failed:", error)
            continue

        print(
            "A* path points:",
            len(path_info["points"])
        )

        # Teacher and AI run independently
        teacher_result = simulate_45(
            "Teacher",
            start,
            goal,
            theta,
            path_info
        )

        ai_result = simulate_45(
            "AI",
            start,
            goal,
            theta,
            path_info
        )

        print("\n===== CLOSED-LOOP RESULTS =====")

        for result in [
            teacher_result,
            ai_result
        ]:

            print(
                f"{result['name']:8s} | "
                f"{result['status']:13s} | "
                f"Time: {result['time']:.2f} s | "
                f"Final: {result['final_position']}"
            )

        # Performance metrics
        metrics = pd.DataFrame([
            metrics_45(
                teacher_result,
                experiment_id
            ),
            metrics_45(
                ai_result,
                experiment_id
            )
        ])

        print("\n===== PERFORMANCE COMPARISON =====")
        display(metrics.round(5))

        all_metrics_45.extend(
            metrics.to_dict("records")
        )

        all_results_45[experiment_id] = {
            "teacher": teacher_result,
            "ai": ai_result,
            "path": path_info
        }

        # Save trajectory plot
        trajectory_file = save_trajectory_plot_45(
            experiment,
            path_info,
            teacher_result,
            ai_result
        )

        # Save side-by-side MP4
        video_file = save_comparison_video_45(
            experiment,
            path_info,
            teacher_result,
            ai_result
        )

        # Save individual time-series data
        teacher_result["records"].to_csv(
            f"step4_5_{experiment_id}_teacher.csv",
            index=False
        )

        ai_result["records"].to_csv(
            f"step4_5_{experiment_id}_ai.csv",
            index=False
        )

finally:

    # Always restore the original obstacles.
    obstacles = ORIGINAL_OBSTACLES_45

    print("\nOriginal obstacle layout restored.")


# ------------------------------------------------------------
# 11. Combined performance summary
# ------------------------------------------------------------
if all_metrics_45:

    summary_45 = pd.DataFrame(
        all_metrics_45
    )

    summary_45.to_csv(
        "step4_5_generalization_summary.csv",
        index=False
    )

    print("\n===== STEP 4-5 FINAL SUMMARY =====")

    display(summary_45.round(5))

    print(
        "\nSaved:",
        "step4_5_generalization_summary.csv"
    )

else:
    print("No experiments completed.")

print("\nSTEP 4-5 FINISHED")

## 実験結果の確認と提出物

- STEP 4-3: `step4_3_imitation_model.pth`、学習曲線・予測比較PNG
- STEP 4-4: `step4_4_teacher_vs_ai.mp4`、比較CSV・軌跡PNG
- STEP 4-5: `step4_5_experiment_1_comparison.mp4`、`step4_5_experiment_2_comparison.mp4`、`step4_5_generalization_summary.csv`

**評価上の留意点：** STEP 4-5の2条件での成功は、あらゆる未知環境への汎化を保証するものではありません。A*の経路計画と、MLPの経路追従制御の役割を区別して説明する必要があります。
